In [1]:
import pm4py
from pm4py.objects.log.importer.xes import factory as xes_importer
from pm4py.statistics.traces.log import case_statistics
from pm4py.algo.filtering.log.attributes import attributes_filter
from pm4py.algo.filtering.log.variants import variants_filter
from pm4py.visualization.process_tree import factory as pt_vis_factory
import pandas as pd

# 1. Wczytanie logu
log = xes_importer.apply("BPI_Challenge_2012.xes")

# 2. Konwersja logu do DataFrame (łatwiej analizować)
df = pm4py.convert_to_dataframe(log)

# 3. Sprawdzenie unikalnych aktywności
print("Unikalne aktywności:", df["concept:name"].unique())

# 4. Sprawdzenie rozkładu rezultatów końcowych
final_events = df.groupby("case:concept:name").tail(1)
print("Wyniki końcowe:", final_events["concept:name"].value_counts())

# 5. Segmentacja przypadków
approved_cases = final_events[final_events["concept:name"] == "A_APPROVED"]["case:concept:name"]
declined_cases = final_events[final_events["concept:name"] == "A_DECLINED"]["case:concept:name"]
cancelled_cases = final_events[final_events["concept:name"] == "A_CANCELLED"]["case:concept:name"]

# 6. Dodanie kolumny długości sprawy
df["time:timestamp"] = pd.to_datetime(df["time:timestamp"])
case_durations = df.groupby("case:concept:name")["time:timestamp"].agg(lambda x: (x.max() - x.min()).total_seconds()/3600)

# 7. Połączenie danych do analizy segmentów
segment_df = pd.DataFrame({
    "duration_hours": case_durations,
    "final_result": final_events.set_index("case:concept:name")["concept:name"],
    "amount_req": final_events.set_index("case:concept:name")["AMOUNT_REQ"]
}).dropna()

print(segment_df.groupby("final_result").describe())

# 8. Wizualizacja (np. boxplot czasu trwania)
import seaborn as sns
import matplotlib.pyplot as plt

sns.boxplot(data=segment_df, x="final_result", y="duration_hours")
plt.title("Case duration by outcome")
plt.ylabel("Duration (hours)")
plt.xlabel("Final outcome")
plt.show()


ImportError: cannot import name 'factory' from 'pm4py.objects.log.importer.xes' (/Users/igor/PyCharmMiscProject/.venv/lib/python3.13/site-packages/pm4py/objects/log/importer/xes/__init__.py)